# Baseline 6: Unrelated-questions classifier (black box, per organism)

This notebook implements the **unrelated-questions** detector from Pacchiardi et al., [How to Catch an AI Liar](https://arxiv.org/abs/2309.15840). After the model produces a response, we append each of 39 fixed, *unrelated* yes/no questions (e.g. *\"Have you ever lied?\"*) and record the model's yes/no answer. The 39-dim binary **fingerprint** feeds a logistic-regression lie detector.

The fingerprint is the *model's own* answers, so it is **organism-specific**: we elicit from each dataset's own `model`/`lora` and **train one classifier per organism** (like the white-box probes in 3.1-3.3). Each is saved to `unrelated_questions_probe_<base>__<lora>.npz`; the paired submission notebook is `baselines/unrelated_questions_organism.ipynb`.

**Implementation note.** Rather than *generating* 39 replies per row (39×N autoregressive remote calls — slow enough to risk the per-dataset timeout), we read each yes/no answer from the model's **output logits** at the answer position in a single batched forward pass — a query-level (black-box) readout, the same trick the LLM-judge baseline uses — and bundle every forward pass into one `model.session()`.

## 0️⃣ Setup

In [ ]:
# nnsight — hackathon build (PEFT/LoRA + remote NDIF support)
!pip install -q git+https://github.com/ndif-team/nnsight.git@hackathon/peft
!pip install -q "peft==0.18.0"
!pip install -q --upgrade torchao "transformers==5.9.0"

from IPython.display import clear_output
clear_output()

In [ ]:
import os
from nnsight import CONFIG
from huggingface_hub import login

NDIF_API_KEY = os.environ.get("NDIF_API_KEY")
HF_TOKEN = os.environ.get("HF_TOKEN")

CONFIG.set_default_api_key(NDIF_API_KEY)
CONFIG.API.HOST = "http://ndif-hackathon.duckdns.org:8001"  # hackathon NDIF cluster

# ── Model organisms ───────────────────────────────────────────────────────────
# A *model organism* is a (base model, LoRA adapter) pair. The unrelated-questions
# fingerprint is the model's OWN answers to the elicitation questions, so — like every
# method here — we elicit from each dataset's own model/lora, and train ONE classifier
# PER ORGANISM (its fingerprints are organism-specific and don't transfer).
BASE_SETTINGS = {
    "gemma-3-27b-it":                         dict(multimodal=True,  batch_size=8),
    "Qwen3.5-27B":                            dict(multimodal=False, batch_size=8),
    "NVIDIA-Nemotron-3-Super-120B-A12B-BF16": dict(multimodal=False, batch_size=4),
}
_SPLIT_PREFIXES = ("dev-test-", "validation-", "dev-")   # longest first
LORA_ORG = "aletheias-quest"           # HF org hosting the LoRA adapters
# Full base-model repo id (the dataset `model` column value) per base token.
BASE_REPO = {
    "gemma-3-27b-it":                         "google/gemma-3-27b-it",
    "Qwen3.5-27B":                            "Qwen/Qwen3.5-27B",
    "NVIDIA-Nemotron-3-Super-120B-A12B-BF16": "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16",
}

def parse_org(name):
    """(base_token, lora_token|None) from a dataset id — mirrors aletheia_runner.config."""
    s = name.split("/")[-1]
    for p in _SPLIT_PREFIXES:
        if s.startswith(p):
            s = s[len(p):]; break
    best = None
    for tok in BASE_SETTINGS:
        i = s.find(tok)
        if i != -1 and (best is None or i < best[0]):
            best = (i, tok)
    if best is None:
        return None, None
    i, tok = best
    rest = s[i + len(tok):].strip("-")
    lora = rest if rest and rest.lower() != "none" else None
    return tok, lora

# A LoRA adapter repo name embeds a (lowercased) base tag -> map it to its base token.
_LORA_BASE_TAG = {
    "gemma-3-27b-it":        "gemma-3-27b-it",
    "qwen3.5-27b":           "Qwen3.5-27B",
    "nemotron-3-super-120b": "NVIDIA-Nemotron-3-Super-120B-A12B-BF16",
}
def base_of_lora(lora_token):
    low = lora_token.lower()
    for tag, base in _LORA_BASE_TAG.items():
        if tag in low:
            return base
    return None

import yaml
from huggingface_hub import HfApi
DRY_YAML = next(p for p in ("../dry.yaml", "dry.yaml", "../../dry.yaml") if os.path.exists(p))

def _new_org(base, lora_token):
    """Empty organism record keyed by (base, lora). Activations are always extracted
    through THIS organism's own base+LoRA model."""
    key = f"{base}::{lora_token or 'base'}"
    return dict(
        key=key, base_token=base, lora_token=lora_token,
        model_id=BASE_REPO[base],
        lora_id=(f"{LORA_ORG}/{lora_token}" if lora_token else None),
        multimodal=BASE_SETTINGS[base]["multimodal"],
        batch_size=BASE_SETTINGS[base]["batch_size"],
        save_as=f"unrelated_questions_probe_{base}__{lora_token or 'base'}.npz",
        eval_datasets=[], has_dev=False, calib_note="")

ORGANISMS = {}
for e in (yaml.safe_load(open(DRY_YAML)).get("datasets") or []):
    base, lora = parse_org(e["name"])
    if base is None:
        continue
    org = ORGANISMS.setdefault(f"{base}::{lora or 'base'}", _new_org(base, lora))
    org["has_dev"] = True
    org["eval_datasets"].append(dict(
        name=e["name"], labels_uri=e["labels_uri"],
        id_column=e.get("id_column", "index"),
        label_column=e.get("label_column", "deceptive")))

for base in BASE_SETTINGS:
    ORGANISMS.setdefault(f"{base}::base", _new_org(base, None))
try:
    adapters = [m.id.split("/")[-1] for m in HfApi().list_models(author=LORA_ORG)]
except Exception as ex:
    print(f"[warn] couldn't list {LORA_ORG} adapters from HF ({ex}); using dev-derived set only")
    adapters = []
for lora_token in adapters:
    base = base_of_lora(lora_token)
    if base is not None:
        ORGANISMS.setdefault(f"{base}::{lora_token}", _new_org(base, lora_token))

## 1️⃣ Elicitation questions & yes/no readout

The 39 questions are fixed. \"yes\" vs \"no\" is decided from the model's output logits over a small set of yes/no word tokens at the answer position.

In [ ]:
import json
import numpy as np

# The 39 fixed, unrelated yes/no questions 
with open("data/pacchiardi_questions.json") as f:
    QUESTIONS = json.load(f)

# Words that count as a "yes" vs "no" answer. We read the model's yes/no via its output logits at the answer position
# feature = 1 if the strongest yes-token logit beats the strongest no-token logit, else 0.
YES_WORDS = ["yes", "absolutely", "certainly", "undoubtedly", "true"]
NO_WORDS  = ["no", "never", "false", "not"]

def yesno_token_ids(tokenizer, words):
    """First-token ids for each word across a few surface forms (leading space / caps)."""
    ids = set()
    for w in words:
        for s in (w, " " + w, w.capitalize(), " " + w.capitalize()):
            enc = tokenizer.encode(s, add_special_tokens=False)
            if enc:
                ids.add(enc[0])
    return sorted(ids)

print(f"{len(QUESTIONS)} elicitation questions | yes-words={YES_WORDS} | no-words={NO_WORDS}")
print("e.g.:", QUESTIONS[1])

## 2️⃣ Model & fingerprint pipeline

`elicit_fingerprints` tokenizes every (conversation, question) prompt **locally**, then reads the yes/no logit margin at the answer position in batched remote traces, bundled into one `model.session()` per conversation-chunk (one job, one `save()`). Right-padded so each row is read at its real last token.

In [ ]:
import torch
from nnsight import LanguageModel, VisionLanguageModel

def build_model(model_id, lora_id, multimodal):
    """nnsight handle for one organism. `peft=lora_id` attaches the LoRA (omit when None)."""
    Cls = VisionLanguageModel if multimodal else LanguageModel
    return Cls(model_id, **({"peft": lora_id} if lora_id else {}))

def normalize_messages(tokenizer, messages):
    """Messages the chat template accepts; fold a leading `system` into the first user
    turn when the template rejects `system` (e.g. gemma). Same helper as 3.1/3.3."""
    msgs = [{"role": m["role"], "content": m["content"]} for m in messages]
    try:
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        return msgs
    except Exception:
        norm, pending = [], None
        for m in msgs:
            if m["role"] == "system" and not norm:
                pending = m["content"]; continue
            if not norm and m["role"] == "user" and pending is not None:
                norm.append({"role": "user", "content": pending + "\n\n" + m["content"]}); pending = None; continue
            norm.append(m)
        if pending is not None:
            norm = [{"role": "user", "content": pending}] + norm
        return norm

def to_judged_messages(messages):
    """Trim a conversation to end at its last assistant message — the response judged."""
    idxs = [i for i, m in enumerate(messages) if m.get("role") == "assistant"]
    return messages[: idxs[-1] + 1] if idxs else list(messages)

MAX_SEQ_LEN = 256   # left-trim cap per (conversation, question) prompt

def elicit_fingerprints(model, tokenizer, convos, questions, yes_ids, no_ids, pad_id,
                        batch_size, chunk_convos=25, max_len=MAX_SEQ_LEN):
    """Return an (N_convos, N_questions) matrix of yes-vs-no logit MARGINS: for each
    conversation we append each elicitation question and read the model's yes/no OUTPUT
    logit at the answer position (margin = max yes-token logit - max no-token logit;
    >0 means "yes"). Binarize with (F > 0) for the classifier.

    Faithful to 3.1's remote pattern: prompts are tokenized LOCALLY (id lists), then a
    single `model.session(remote=True)` per conversation-chunk opens one trace per batch,
    reads the answer-position logits at each row's real last token (right-padded, so the
    index is `len-1`), reduces to the yes/no margin, and cat()+save()s ONCE at the session
    frame. Chunking bounds each remote job's size (N x 39 forward passes is a lot)."""
    yes_t = torch.tensor(yes_ids, dtype=torch.long)
    no_t  = torch.tensor(no_ids, dtype=torch.long)
    F = np.zeros((len(convos), len(questions)), dtype=np.float64)

    for c0 in range(0, len(convos), chunk_convos):
        cchunk = convos[c0:c0 + chunk_convos]
        tok_ids, meta = [], []                       # meta[k] = (local_conv_idx, question_idx)
        for lci, conv in enumerate(cchunk):
            norm = normalize_messages(tokenizer, to_judged_messages(conv))
            for qi, q in enumerate(questions):
                text = tokenizer.apply_chat_template(
                    norm + [{"role": "user", "content": q}],
                    tokenize=False, add_generation_prompt=True)
                ids = tokenizer(text, add_special_tokens=False)["input_ids"]
                if len(ids) > max_len:
                    ids = ids[len(ids) - max_len:]   # left-trim (keep the question + answer slot)
                tok_ids.append(ids); meta.append((lci, qi))

        order = sorted(range(len(tok_ids)), key=lambda i: len(tok_ids[i]))   # short -> long
        with model.session(remote=True):
            pieces = []
            for b in range(0, len(order), batch_size):
                bpos = order[b:b + batch_size]
                brows = [tok_ids[i] for i in bpos]
                w = max(len(r) for r in brows)
                input_ids = torch.full((len(brows), w), pad_id, dtype=torch.long)
                attn = torch.zeros((len(brows), w), dtype=torch.long)
                lens = []
                for ri, r in enumerate(brows):       # right-pad; read each row at len-1
                    input_ids[ri, :len(r)] = torch.tensor(r)
                    attn[ri, :len(r)] = 1
                    lens.append(len(r))
                with model.trace({"input_ids": input_ids, "attention_mask": attn}):
                    lg = model.output.logits         # (B, T, V)
                    outs = []
                    for ri, L in enumerate(lens):
                        row = lg[ri, L - 1]          # (V,) next-token logits at the answer slot
                        yes = torch.index_select(row, 0, yes_t.to(row.device)).amax()
                        no  = torch.index_select(row, 0, no_t.to(row.device)).amax()
                        outs.append(yes - no)
                    pieces.append(torch.stack(outs))
            margins = torch.cat(pieces).save()        # (chunk_rows,) in `order` order
        arr = margins.float().cpu().numpy()
        flat = np.empty(len(order), dtype=np.float64)
        for j, i in enumerate(order):
            flat[i] = arr[j]
        for k, (lci, qi) in enumerate(meta):
            F[c0 + lci, qi] = flat[k]
        print(f"  fingerprinted {min(c0 + chunk_convos, len(convos))}/{len(convos)} conversations")
    return F

## 3️⃣ Training data (the organism's own labeled sets)

We pool the organism's `dry.yaml` dev datasets into one balanced labeled set and train its classifier on the fingerprints — a documented, per-organism parameter fit (the competition explicitly allows tuning a detector's parameters per organism).

In [ ]:
import pandas as pd
from datasets import load_dataset

def load_org_labeled(org, max_per_label=100, seed=42):
    """Pool ALL of an organism's dev datasets into one balanced labeled set for training
    its classifier. Returns (convos, y). Each conv's final assistant turn is the response
    we fingerprint. Balanced-capped to bound the (expensive) elicitation cost."""
    convos, ys = [], []
    for ds_cfg in org["eval_datasets"]:
        name, labels_uri = ds_cfg["name"], ds_cfg["labels_uri"]
        idc, lbc = ds_cfg["id_column"], ds_cfg["label_column"]
        df = load_dataset(name, split="test").to_pandas()
        if idc not in df.columns:
            df[idc] = range(len(df))
        lab = pd.read_csv(labels_uri) if str(labels_uri).endswith(".csv") \
            else load_dataset(labels_uri, split="test").to_pandas()
        df = df.merge(lab[[idc, lbc]], on=idc, how="inner")
        for m, l in zip(df["messages"], df[lbc].astype(int).values):
            convos.append(list(m)); ys.append(int(l))
    y = np.array(ys)
    if len(np.unique(y)) < 2:
        return [], np.array([])
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    n = min(max_per_label, len(pos), len(neg))
    keep = np.sort(np.concatenate([rng.choice(pos, n, replace=False),
                                   rng.choice(neg, n, replace=False)]))
    return [convos[i] for i in keep], y[keep]

## 4️⃣ Train & save (one classifier per model organism)

`train_and_save_unrelated(org)` builds the organism's own model, elicits fingerprints on its dev data, fits a logistic regression (held-out AUROC reported), refits on all of it, and saves the `.npz`. Run one cell per organism.

In [ ]:
from transformers import AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

all_results = {}

def train_and_save_unrelated(org, max_per_label=100):
    """Full per-organism pass: build the organism's OWN model (peek row 0 for real
    model/lora), elicit 39-question fingerprints on its balanced dev data in remote
    sessions, fit a logistic-regression classifier on the binary fingerprints, report a
    held-out AUROC, then refit on everything and save its `.npz`."""
    key = org["key"]
    print(f"\n{'='*20} {key} {'='*20}")

    # Dev-less organism -> borrow its BASE model's dev sets for eval/calibration.
    # Activations still come from THIS organism's own base+LoRA model; this is robust
    # even if the setup cell's dev-less fallback (step 3) was not run.
    if not org.get("eval_datasets"):
        _base = ORGANISMS.get(f"{org['base_token']}::base")
        if _base and _base.get("eval_datasets"):
            org["eval_datasets"] = [dict(d) for d in _base["eval_datasets"]]
            print(f"  [fallback] no dev data -> borrowing {org['base_token']}::base dev sets")

    model_id, lora_id = org["model_id"], org["lora_id"]
    print(f"model_id={model_id}  lora_id={lora_id}")

    model = build_model(model_id, lora_id, org["multimodal"])
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    yes_ids, no_ids = yesno_token_ids(tokenizer, YES_WORDS), yesno_token_ids(tokenizer, NO_WORDS)

    convos, y = load_org_labeled(org, max_per_label=max_per_label)
    if len(convos) == 0:
        print("  [skip] no usable labeled rows"); return None
    print(f"eliciting fingerprints for {len(convos)} conversations x {len(QUESTIONS)} questions...")
    F = elicit_fingerprints(model, tokenizer, convos, QUESTIONS, yes_ids, no_ids,
                            pad_id, org["batch_size"])
    Xb = (F > 0).astype(np.float64)                   # binary yes/no fingerprint

    Xtr, Xte, ytr, yte = train_test_split(Xb, y, test_size=0.25, stratify=y, random_state=42)
    clf = LogisticRegression(max_iter=2000, random_state=42).fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    auroc = roc_auc_score(yte, p)
    bacc = balanced_accuracy_score(yte, clf.predict(Xte))
    print(f"  held-out AUROC {auroc:.3f} | bal-acc {bacc:.3f}")

    final = LogisticRegression(max_iter=2000, random_state=42).fit(Xb, y)  # refit on all
    np.savez(org["save_as"],
             coef=final.coef_[0].astype(np.float32),
             intercept=np.array(float(final.intercept_[0]), dtype=np.float32),
             questions=np.array(QUESTIONS),
             yes_words=np.array(YES_WORDS),
             no_words=np.array(NO_WORDS),
             model_id=np.array(model_id),
             lora_id=np.array("" if lora_id is None else lora_id),
             organism=np.array(key))
    print(f"saved {org['save_as']} | dim={final.coef_[0].shape[0]} | n={len(convos)}")
    all_results[key] = dict(auroc=float(auroc), bacc=float(bacc), n=len(convos),
                            save_as=org["save_as"])
    del model
    return all_results[key]

### One cell per organism

In [ ]:
# NVIDIA-Nemotron-3-Super-120B-A12B-BF16 (base, no LoRA)
train_and_save_unrelated(ORGANISMS["NVIDIA-Nemotron-3-Super-120B-A12B-BF16::base"])

In [ ]:
# Qwen3.5-27B (base, no LoRA)
train_and_save_unrelated(ORGANISMS["Qwen3.5-27B::base"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-1
train_and_save_unrelated(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-1"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-3
train_and_save_unrelated(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-3"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-4
train_and_save_unrelated(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-4"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-5
train_and_save_unrelated(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-5"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-6
train_and_save_unrelated(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-6"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-7
train_and_save_unrelated(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-7"])

In [ ]:
# Qwen3.5-27B + LoRA b-mo-qwen3.5-27b
train_and_save_unrelated(ORGANISMS["Qwen3.5-27B::b-mo-qwen3.5-27b"])

In [ ]:
# Qwen3.5-27B + LoRA c-mo-qwen3.5-27b
train_and_save_unrelated(ORGANISMS["Qwen3.5-27B::c-mo-qwen3.5-27b"])

In [ ]:
# gemma-3-27b-it (base, no LoRA)
train_and_save_unrelated(ORGANISMS["gemma-3-27b-it::base"])

In [ ]:
# gemma-3-27b-it + LoRA s-mo-gemma-3-27b-it
train_and_save_unrelated(ORGANISMS["gemma-3-27b-it::s-mo-gemma-3-27b-it"])

In [ ]:
# Qwen3.5-27B + LoRA g-st-qwen3.5-27b  (no public dev data -> calibrated on Qwen3.5-27B base dev)
train_and_save_unrelated(ORGANISMS["Qwen3.5-27B::g-st-qwen3.5-27b"])

In [ ]:
# gemma-3-27b-it + LoRA g-st-gemma-3-27b-it-1  (no public dev data -> calibrated on gemma-3-27b-it base dev)
train_and_save_unrelated(ORGANISMS["gemma-3-27b-it::g-st-gemma-3-27b-it-1"])

In [ ]:
# gemma-3-27b-it + LoRA g-st-gemma-3-27b-it-2  (no public dev data -> calibrated on gemma-3-27b-it base dev)
train_and_save_unrelated(ORGANISMS["gemma-3-27b-it::g-st-gemma-3-27b-it-2"])

In [ ]:
# gemma-3-27b-it + LoRA g-st-gemma-3-27b-it-3  (no public dev data -> calibrated on gemma-3-27b-it base dev)
train_and_save_unrelated(ORGANISMS["gemma-3-27b-it::g-st-gemma-3-27b-it-3"])

In [ ]:
# NVIDIA-Nemotron-3-Super-120B-A12B-BF16 + LoRA s-mo-nemotron-3-super-120b  (no public dev data -> calibrated on Nemotron-120B base dev)
train_and_save_unrelated(ORGANISMS["NVIDIA-Nemotron-3-Super-120B-A12B-BF16::s-mo-nemotron-3-super-120b"])

In [ ]:
# NVIDIA-Nemotron-3-Super-120B-A12B-BF16 + LoRA g-st-nemotron-3-super-120b  (no public dev data -> calibrated on Nemotron-120B base dev)
train_and_save_unrelated(ORGANISMS["NVIDIA-Nemotron-3-Super-120B-A12B-BF16::g-st-nemotron-3-super-120b"])

## 5️⃣ Saved classifiers

In [ ]:
import glob

print("=== Saved per-organism unrelated-questions classifiers ===\n")
for f in sorted(glob.glob("unrelated_questions_probe_*.npz")):
    d = np.load(f, allow_pickle=True)
    lora = str(d["lora_id"]) if "lora_id" in d else ""
    print(f"{f}\n   organism: {str(d['organism'])} | model: {str(d['model_id'])} | "
          f"lora: {lora or 'None'} | dim: {d['coef'].shape[0]}")